# Content-based Filtering

Recommending items based on their features and a user's taste profile — no other users involved.

## Imports and Configuration

In [2]:
import sys
sys.path.append('../../datasets') # two levels up from the notebook

In [3]:
import pandas as pd
from data_loader import DataLoader

## Load and Explore the Data

In [90]:
loader = DataLoader()

ratings_df = loader.load("ml-latest-small/ratings.csv")
movies_df = loader.load("ml-latest-small/movies.csv")

In [91]:
ratings_df.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [92]:
movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [93]:
print(f"Ratings: {ratings_df.shape[0]} rows")
print(f"Movies: {movies_df.shape[0]} unique")
print(f"Users: {ratings_df["userId"].nunique()} unique")

Ratings: 100836 rows
Movies: 9742 unique
Users: 610 unique


## Prepare Movie data

Split the values in the Genres column into a list of Genres to simplify future use.

In [94]:
movies_df["genres"] = movies_df["genres"].apply(lambda x: x.split("|"))
movies_df.head(2)

,movieId,title,genres
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]"
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]"


Iterate through movies_df, then append the movie genres as columns of 1s or 0s.  
1 if that column contains movies in the genre at the present index and 0 if not.

In [95]:
movies_with_genres = movies_df.copy(deep=True)

for index, row in movies_df.iterrows():
    for genre in row["genres"]:
        movies_with_genres.at[index, genre] = 1

movies_with_genres.head(3)

,movieId,title,genres,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,...,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir,(no genres listed)
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1.0,1.0,1.0,1.0,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",1.0,NaN,1.0,NaN,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",NaN,NaN,NaN,1.0,NaN,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Fill in the NaN values with 0 to show that a movie doesn't have that column's genre

In [96]:
movies_with_genres = movies_with_genres.fillna(0)
movies_with_genres.head(2)

,movieId,title,genres,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,...,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir,(no genres listed)
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Prepare Rating Data

In [97]:
# We do not need the timestamp column
ratings_df.drop(columns="timestamp", inplace=True)
ratings_df.head(2)

,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0


In [98]:
# Check for null values
ratings_df.isna().sum()

userId     0
movieId    0
rating     0
dtype: int64

## Content-Based Recommender System

In [99]:
# Lets generate recommendations for user with id 2
user_2_preferences = ratings_df[ratings_df["userId"] == 2]
user_2_preferences.count()

userId     29
movieId    29
rating     29
dtype: int64

In [100]:
# Now add the movie details for the movies interacted by the user
user2_movie_details = user_2_preferences.merge(movies_df, on="movieId")
user2_movie_details

,userId,movieId,rating,title,genres
0,2,318,3.0,"Shawshank Redemption, The (1994)","[Crime, Drama]"
1,2,333,4.0,Tommy Boy (1995),[Comedy]
2,2,1704,4.5,Good Will Hunting (1997),"[Drama, Romance]"
3,2,3578,4.0,Gladiator (2000),"[Action, Adventure, Drama]"
4,2,6874,4.0,Kill Bill: Vol. 1 (2003),"[Action, Crime, Thriller]"
5,2,8798,3.5,Collateral (2004),"[Action, Crime, Drama, Thriller]"
6,2,46970,4.0,Talladega Nights: The Ballad of Ricky Bobby (2...,"[Action, Comedy]"
7,2,48516,4.0,"Departed, The (2006)","[Crime, Drama, Thriller]"
8,2,58559,4.5,"Dark Knight, The (2008)","[Action, Crime, Drama, IMAX]"
9,2,60756,5.0,Step Brothers (2008),[Comedy]


In [137]:
user2_movies_with_genres = movies_with_genres[movies_with_genres["movieId"].isin(user_2_preferences["movieId"])]
user2_movies_with_genres = user2_movies_with_genres.fillna(0).reset_index()

In [138]:
user2_movies_with_genres = user2_movies_with_genres.drop(columns=["index", "movieId", "title", "genres"])
user2_movies_with_genres

,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,Action,Crime,Thriller,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir,(no genres listed)
0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
9,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Building Profile of User 2

In [34]:
print("User 2 genre matrix shape: ", user2_movies_with_genres.shape)
print("USer 2 preferences shape: ", user_2_preferences.shape)

User 2 genre matrix shape:  (29, 20)
USer 2 preferences shape:  (29, 3)


In [ ]:
# Perform dot product to get a per genre weighted preference list for the user

user2_profile = user2_movies_with_genres.T.dot(user2_movie_details.rating)
user2_profile

Adventure             12.5
Animation              0.0
Children               0.0
Comedy                28.0
Fantasy                0.0
Romance                4.5
Drama                 66.0
Action                43.5
Crime                 38.0
Thriller              37.0
Horror                 3.0
Mystery                8.0
Sci-Fi                15.5
War                    4.5
Musical                0.0
Documentary           13.0
IMAX                  15.0
Western                3.5
Film-Noir              0.0
(no genres listed)     0.0
dtype: float64

Looking at the profile, one can easily say that User 2 loves movies of `Drama` followed by `Action` and so on. 

In [36]:
movies_with_genres = movies_with_genres.set_index(movies_with_genres["movieId"])
movies_with_genres.head()

,movieId,title,genres,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,...,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir,(no genres listed)
movieId,,,,,,,,,,,,,,,,,,,,,
1,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,Jumanji (1995),"[Adventure, Children, Fantasy]",1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,Grumpier Old Men (1995),"[Comedy, Romance]",0.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",0.0,0.0,0.0,1.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,5,Father of the Bride Part II (1995),[Comedy],0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [37]:
# Remove the columns no longer needed

movies_with_genres = movies_with_genres.drop(columns=["movieId", "title", "genres"])
movies_with_genres.head()

,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,Action,Crime,Thriller,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir,(no genres listed)
movieId,,,,,,,,,,,,,,,,,,,,
1,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [41]:
print("User 2 profile: ", user2_profile.shape)
print("Movies with genres: ", movies_with_genres.shape)

User 2 profile:  (20,)
Movies with genres:  (9742, 20)


In [ ]:

# Perform a dot product of the genre matrix with the user profile to get the recommendations, normalize it by dividing by the total profile weight
recommendation_list = movies_with_genres.dot(user2_profile) / user2_profile.sum()

# Sort the list in descending order
recommendation_list = recommendation_list.sort_values(ascending=False)
recommendation_list[:20]

movieId
81132     0.820205
79132     0.763699
4719      0.743151
7235      0.738014
145       0.727740
5027      0.727740
1432      0.727740
7007      0.727740
20        0.727740
5628      0.727740
198       0.712329
26701     0.712329
459       0.690068
49530     0.690068
2985      0.684932
519       0.684932
86644     0.683219
6016      0.674658
165347    0.659247
5388      0.659247
dtype: float64

In [ ]:
# Copy the movies data and index by movieId
movies_indexed = movies_df.copy(deep=True)
movies_indexed = movies_indexed.set_index(movies_indexed.movieId)
movies_indexed.head()

,movieId,title,genres
movieId,,,
1,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]"
2,2,Jumanji (1995),"[Adventure, Children, Fantasy]"
3,3,Grumpier Old Men (1995),"[Comedy, Romance]"
4,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]"
5,5,Father of the Bride Part II (1995),[Comedy]


In [ ]:
# Extract top k movieIds to recommend
k = 20
top_k_recommendation_list = list(recommendation_list.index[:k])
top_k_recommendation_list

[81132,
 79132,
 4719,
 7235,
 145,
 5027,
 1432,
 7007,
 20,
 5628,
 198,
 26701,
 459,
 49530,
 2985,
 519,
 86644,
 6016,
 165347,
 5388]

In [ ]:
# Get the movies corresponding to the indices in the movieId indexed data
recommendations = movies_indexed.loc[top_k_recommendation_list]
recommendations

,movieId,title,genres
movieId,,,
81132,81132,Rubber (2010),"[Action, Adventure, Comedy, Crime, Drama, Film..."
79132,79132,Inception (2010),"[Action, Crime, Drama, Mystery, Sci-Fi, Thrill..."
4719,4719,Osmosis Jones (2001),"[Action, Animation, Comedy, Crime, Drama, Roma..."
7235,7235,Ichi the Killer (Koroshiya 1) (2001),"[Action, Comedy, Crime, Drama, Horror, Thriller]"
145,145,Bad Boys (1995),"[Action, Comedy, Crime, Drama, Thriller]"
5027,5027,Another 48 Hrs. (1990),"[Action, Comedy, Crime, Drama, Thriller]"
1432,1432,Metro (1997),"[Action, Comedy, Crime, Drama, Thriller]"
7007,7007,"Last Boy Scout, The (1991)","[Action, Comedy, Crime, Drama, Thriller]"
20,20,Money Train (1995),"[Action, Comedy, Crime, Drama, Thriller]"


The following function combines the whole content-based recommendation process

In [142]:
def get_recommendations(userId, k=10):
    # Get Movie preferences of the User
    user_preferences = ratings_df[ratings_df["userId"] == userId]
    print(
        f"User {userId} has {user_preferences["movieId"].count()} interactions in the data."
    )

    # Movies interacted by the user
    user_movie_details = user_preferences.merge(movies_df, on="movieId")
    print(f"\nMovies interacted by User {userId}:\n")
    print(user_movie_details[["title", "rating"]])

    user_movies_and_genres = movies_with_genres[
        movies_with_genres["movieId"].isin(user_preferences["movieId"])
    ]
    # Fill NaN values
    user_movies_and_genres = user_movies_and_genres.fillna(0).reset_index()
    
    user_movies_and_genres = user_movies_and_genres.drop(columns=["index", "movieId", "title", "genres"])
    
    print("\nUser genre matrix shape: ", user_movies_and_genres.shape)
    print("User preferences shape: ", user_preferences.shape)
    
    # Perform dot product between user genre matrix and user preferences to get per-genre user preference
    user_profile = user_movies_and_genres.T.dot(user_preferences.rating.reset_index())
    user_profile = user_profile.drop(columns=["index"])
    print("\nUser Profile: \n", user_profile)
    
    movies_with_id_index_and_genres = movies_with_genres.set_index(movies_with_genres["movieId"])
    movies_with_id_index_and_genres = movies_with_id_index_and_genres.drop(columns=["movieId", "title", "genres"])
    
    print("\nUser Profile: ", user_profile.shape)
    print("Movies with genres: ", movies_with_id_index_and_genres.shape)
    
    # Perform a dot product of the genre matrix with the user profile to get the recommendations, normalize it by dividing by the total profile weight
    all_recommendations = movies_with_id_index_and_genres.dot(user_profile) / user_profile.sum()
    
    # Sort the list in descending order
    all_recommendations = all_recommendations.sort_values(by="rating", ascending=False)
    
    # Create a copy of the original dataframe and index by movieId
    movies_id_indexed = movies_df.copy(deep=True)
    movies_id_indexed = movies_id_indexed.set_index(movies_id_indexed.movieId)
    
    # Extract top-k moviesIds
    recommended_movie_ids = list(all_recommendations.index[:k])
    
    # Find movies corresponding to top-k movieIds
    recommendations = movies_id_indexed.loc[recommended_movie_ids]
    print("-"* 100)
    print(f"\nRecommendations for user {userId}")
    print(recommendations[["title", "genres"]])

get_recommendations(userId=2, k=10)

User 2 has 29 interactions in the data.

Movies interacted by User 2:

                                                title  rating
0                    Shawshank Redemption, The (1994)     3.0
1                                    Tommy Boy (1995)     4.0
2                            Good Will Hunting (1997)     4.5
3                                    Gladiator (2000)     4.0
4                            Kill Bill: Vol. 1 (2003)     4.0
5                                   Collateral (2004)     3.5
6   Talladega Nights: The Ballad of Ricky Bobby (2...     4.0
7                                Departed, The (2006)     4.0
8                             Dark Knight, The (2008)     4.5
9                                Step Brothers (2008)     5.0
10                        Inglourious Basterds (2009)     4.5
11                                  Zombieland (2009)     3.0
12                              Shutter Island (2010)     4.0
13                  Exit Through the Gift Shop (2010)     3.0